# NB11 — Clustering de Municipios + Análisis de Sesgos (Unidad II)**Cobertura del syllabus — Unidad II:**- Técnicas para agrupamiento (K-Means, DBSCAN, Hierarchical)- Aplicaciones del agrupamiento (perfilado de municipios cafeteros)- Herramientas para análisis de sesgos (Fairlearn, métricas de equidad)**Objetivo:** identificar perfiles agroclimáticos y verificar que los modelos no tengan sesgo geográfico (Huila vs Antioquia vs Nariño).

In [ ]:
import warnings, jsonfrom pathlib import Pathimport numpy as np, pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.cluster import KMeans, DBSCAN, AgglomerativeClusteringfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.manifold import TSNEfrom sklearn.metrics import silhouette_score, davies_bouldin_scorewarnings.filterwarnings('ignore')RNG=42; np.random.seed(RNG)PROJECT=Path('..').resolve()DIR_FIG=PROJECT/'05_resultados'/'figuras'; DIR_FIG.mkdir(parents=True,exist_ok=True)DIR_TAB=PROJECT/'05_resultados'/'tablas'; DIR_TAB.mkdir(parents=True,exist_ok=True)

## 1. Carga del dataset municipal

In [ ]:
arch=PROJECT/'01_datos'/'procesados'/'master_cafe_municipal_mensual.csv'arch_2da=PROJECT.parent/'IA_Segunda_Entrega'/'datasets'/'master_cafe_semestral.csv'df=pd.read_csv(arch if arch.exists() else arch_2da)print('Shape:',df.shape)df.head()

## 2. Perfilado por municipio

In [ ]:
# Agregamos por municipio (o departamento si municipio no existe)key='codigo_dane' if 'codigo_dane' in df.columns else 'departamento' if 'departamento' in df.columns else 'cod_dpto_mun'features_clima=[c for c in df.columns if any(s in c.lower() for s in ['temp','precip','et0','ndvi','altitud'])]features_econ=[c for c in df.columns if any(s in c.lower() for s in ['precio','rendimiento','produccion','area'])]features=features_clima+features_econfeatures=[c for c in features if df[c].dtype in (np.float64,np.int64,'float64','int64')]print(f'Features: {len(features)}')perfil=df.groupby(key)[features].mean().dropna()print('Municipios/Dptos perfilados:',len(perfil))perfil.head()

## 3. K-Means con análisis de codo y silhouette

In [ ]:
scaler=StandardScaler().fit(perfil)X=scaler.transform(perfil)inertias=[]; silhouettes=[]; ks=range(2,9)for k in ks:    km=KMeans(n_clusters=k,random_state=RNG,n_init=10).fit(X)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X,km.labels_))fig,axs=plt.subplots(1,2,figsize=(12,4))axs[0].plot(list(ks),inertias,'o-'); axs[0].set_title('Codo (Inercia)'); axs[0].set_xlabel('k')axs[1].plot(list(ks),silhouettes,'o-',color='orange'); axs[1].set_title('Silhouette'); axs[1].set_xlabel('k')plt.tight_layout(); plt.savefig(DIR_FIG/'NB11_codo_silhouette.png',dpi=120); plt.show()k_opt=ks[np.argmax(silhouettes)]print(f'k óptimo: {k_opt}')

## 4. K-Means + DBSCAN + Hierarchical

In [ ]:
km=KMeans(n_clusters=k_opt,random_state=RNG,n_init=10).fit(X)db=DBSCAN(eps=1.5,min_samples=3).fit(X)hc=AgglomerativeClustering(n_clusters=k_opt).fit(X)resultados=pd.DataFrame({'kmeans':km.labels_,'dbscan':db.labels_,'hierarchical':hc.labels_},index=perfil.index)resultados.to_csv(DIR_TAB/'NB11_clusters.csv')print('K-Means silhouette:',silhouette_score(X,km.labels_))if len(set(db.labels_))>1: print('DBSCAN silhouette:',silhouette_score(X,db.labels_))print('Hierarchical silhouette:',silhouette_score(X,hc.labels_))print('Davies-Bouldin (KM):',davies_bouldin_score(X,km.labels_))

## 5. PCA + t-SNE para visualización

In [ ]:
pca=PCA(n_components=2).fit(X); X_pca=pca.transform(X)tsne=TSNE(n_components=2,random_state=RNG,perplexity=min(15,len(X)-1)).fit_transform(X)fig,axs=plt.subplots(1,2,figsize=(14,5))sc1=axs[0].scatter(X_pca[:,0],X_pca[:,1],c=km.labels_,cmap='viridis',s=80,edgecolor='k')axs[0].set_title(f'PCA · K-Means k={k_opt}'); axs[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')axs[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')sc2=axs[1].scatter(tsne[:,0],tsne[:,1],c=km.labels_,cmap='viridis',s=80,edgecolor='k')axs[1].set_title('t-SNE · K-Means')plt.tight_layout(); plt.savefig(DIR_FIG/'NB11_proyecciones.png',dpi=120); plt.show()

## 6. Caracterización de clústeres

In [ ]:
perfil['cluster_km']=km.labels_caract=perfil.groupby('cluster_km').mean().Tcaract.to_csv(DIR_TAB/'NB11_caracterizacion.csv')print(caract)

## 7. Análisis de sesgos con Fairlearn

In [ ]:
# Cargar predicciones del modelo de rendimiento (NB02 o NB09)import pickletry:    from fairlearn.metrics import MetricFrame, demographic_parity_difference, selection_rate    has_fl=Trueexcept ImportError:    print('pip install fairlearn'); has_fl=False# Si tenemos predicciones reales, evaluamos sesgo por departamentopred_path=PROJECT/'05_resultados'/'tablas'/'predicciones_test_rendimiento.csv'if pred_path.exists() and has_fl:    pred=pd.read_csv(pred_path)    if 'departamento' in pred.columns:        from sklearn.metrics import mean_absolute_error        mf=MetricFrame(metrics=mean_absolute_error,            y_true=pred['real'],y_pred=pred['predicho'],            sensitive_features=pred['departamento'])        print('MAE por departamento:'); print(mf.by_group)        diff=mf.difference(method='between_groups')        print(f'\nDiferencia de error entre grupos: {diff:.4f}')else:    print('Sin predicciones disponibles. Ejecuta NB02 o NB09 primero.')    print('Demostración con datos sintéticos:')    np.random.seed(RNG); n=300    sintetico=pd.DataFrame({'departamento':np.random.choice(['Huila','Antioquia','Nariño'],n),        'real':np.random.normal(1.0,0.3,n),'predicho':np.random.normal(0.95,0.3,n)})    if has_fl:        from sklearn.metrics import mean_absolute_error as mae        mf=MetricFrame(metrics=mae,y_true=sintetico['real'],y_pred=sintetico['predicho'],                       sensitive_features=sintetico['departamento'])        print('MAE por dpto (demo):'); print(mf.by_group)

## 8. Mitigación de sesgo: reweighing por clase

In [ ]:
# Para CNN: la clase 'Alto' dominaba 53% en CALIBRO. Aquí mostramos# cómo calcular pesos por clase para entrenar con balanceoclases_demo=['Sano','Roya','Gotera','Cercospora','Phoma','Miner']counts=pd.Series([2500,3000,800,1200,500,2000],index=clases_demo)total=counts.sum(); n_cls=len(counts)weights=total/(n_cls*counts)print('Pesos por clase (para class_weight={...} en model.fit):')print(weights.round(3))plt.figure(figsize=(8,4))weights.plot.bar(color='steelblue'); plt.title('Pesos balanceados por clase')plt.ylabel('weight'); plt.tight_layout()plt.savefig(DIR_FIG/'NB11_weights_balanceo.png',dpi=120); plt.show()

## 9. Conclusiones — Unidad II (clustering + sesgos)- 3 algoritmos de agrupamiento aplicados (K-Means, DBSCAN, Hierarchical)- Visualización 2D con PCA y t-SNE- Caracterización de clústeres por temperatura, precipitación, rendimiento- Análisis de sesgo geográfico con Fairlearn (MetricFrame)- Mitigación: reweighing para datasets desbalanceados**Insight:** los municipios cafeteros se agrupan en perfiles agroclimáticos distintos, y el modelo de rendimiento puede tener sesgos hacia los departamentos sobre-representados — Fairlearn permite cuantificarlo.